# 08 Run Static Allocator Baseline v1

This notebook is a thin execution wrapper around `run_static_allocator_baseline_v1.py`. The production logic lives in the Python script; this notebook only sets parameters, runs the script, and previews the generated outputs.

Run this notebook from the existing `portfolio_allocation_rl` conda environment.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path(
    "/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl"
).resolve()
SCRIPT_PATH = PROJECT_ROOT / "run_static_allocator_baseline_v1.py"

assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
assert SCRIPT_PATH.exists(), f"Allocator script not found: {SCRIPT_PATH}"

PROJECT_ROOT, SCRIPT_PATH

(PosixPath('/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl'),
 PosixPath('/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/run_static_allocator_baseline_v1.py'))

## Configuration

Edit these values to run alternative baseline settings, such as 20 bps transaction costs or a max-weight cap.

In [ ]:
config = {
    "project_root": PROJECT_ROOT,
    "pred_file": "data/prediction/fm_oos_predictions.parquet",
    "risk_dir": "data/risk/risk_cov_npz",
    "risk_meta_file": "data/risk/risk_monthly_metadata.csv",
    "returns_file": "data/panel/monthly_stock_panel_basic_full.parquet",
    "outdir": "data/allocator",
    "lambda_risk": 10.0,
    "tau_turnover": 0.001,
    "cost_bps": 10.0,
    "solver": "CLARABEL",
    "max_weight": None,
    "start_month": None,
    "end_month": None,
    "warm_start": "true",
    "log_level": "INFO",
}

config

{'project_root': PosixPath('/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl'),
 'pred_file': 'data/prediction/fm_oos_predictions.parquet',
 'risk_dir': 'data/risk/risk_cov_npz',
 'risk_meta_file': 'data/risk/risk_monthly_metadata.csv',
 'returns_file': 'data/panel/monthly_stock_panel_basic_full.parquet',
 'outdir': 'data/allocator',
 'lambda_risk': 10.0,
 'tau_turnover': 0.001,
 'cost_bps': 10.0,
 'solver': 'CLARABEL',
 'max_weight': None,
 'start_month': None,
 'end_month': None,
 'warm_start': 'true',
 'log_level': 'INFO'}

## Run Allocator Script

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--project-root",
    str(config["project_root"]),
    "--pred-file",
    config["pred_file"],
    "--risk-dir",
    config["risk_dir"],
    "--risk-meta-file",
    config["risk_meta_file"],
    "--returns-file",
    config["returns_file"],
    "--outdir",
    config["outdir"],
    "--lambda-risk",
    str(config["lambda_risk"]),
    "--tau-turnover",
    str(config["tau_turnover"]),
    "--cost-bps",
    str(config["cost_bps"]),
    "--solver",
    config["solver"],
    "--warm-start",
    config["warm_start"],
    "--log-level",
    config["log_level"],
]

if config["max_weight"] is not None:
    cmd.extend(["--max-weight", str(config["max_weight"])])
if config["start_month"] is not None:
    cmd.extend(["--start-month", str(config["start_month"])])
if config["end_month"] is not None:
    cmd.extend(["--end-month", str(config["end_month"])])

print(" ".join(cmd))
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)

print(result.stdout)
if result.stderr:
    print(result.stderr)

result.check_returncode()

/opt/anaconda3/envs/portfolio_allocation_rl/bin/python /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/run_static_allocator_baseline_v1.py --project-root /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl --pred-file data/prediction/fm_oos_predictions.parquet --risk-dir data/risk/risk_cov_npz --risk-meta-file data/risk/risk_monthly_metadata.csv --returns-file data/panel/monthly_stock_panel_basic_full.parquet --outdir data/allocator --lambda-risk 10.0 --tau-turnover 0.001 --cost-bps 10.0 --solver CLARABEL --warm-start true --log-level INFO

2026-04-28 20:39:00 INFO run_static_allocator_baseline_v1 - Using pyarrow 20.0.0 for parquet IO.
2026-04-28 20:39:00 INFO run_static_allocator_baseline_v1 - Loading prediction file from /Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/prediction/fm_oos_predictions.parquet.
2026-04-28 20:39:00 INFO run_static_allocator_baseline_v1 - Kept 100,092 of 100,092 prediction rows afte

## Output Paths

In [ ]:
outdir = PROJECT_ROOT / config["outdir"]

weights_path = outdir / "static_allocator_weights.parquet"
backtest_path = outdir / "static_allocator_backtest.csv"
summary_path = outdir / "static_allocator_summary.txt"
cumret_plot = outdir / "static_allocator_cumret.png"
turnover_plot = outdir / "static_allocator_turnover.png"
n_assets_plot = outdir / "static_allocator_n_assets.png"

for path in [
    weights_path,
    backtest_path,
    summary_path,
    cumret_plot,
    turnover_plot,
    n_assets_plot,
]:
    print(path, "exists=" + str(path.exists()))

/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/allocator/static_allocator_weights.parquet exists=True
/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/allocator/static_allocator_backtest.csv exists=True
/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/allocator/static_allocator_summary.txt exists=True
/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/allocator/static_allocator_cumret.png exists=True
/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/allocator/static_allocator_turnover.png exists=True
/Users/lava/Documents/Research/GeorgeMichailidis/portfolio_allocation_rl/data/allocator/static_allocator_n_assets.png exists=True


## Preview Backtest Results

In [ ]:
backtest = pd.read_csv(backtest_path, parse_dates=["month_end"])
backtest.head()

,month_end,n_assets,solver_status,objective_value,turnover,gross_return,cost,net_return,cumulative_nav
0,2005-12-31,445,optimal,0.005557,1.774144,0.013854,0.001774,0.012080,1.012080
1,2006-01-31,443,optimal,0.006916,0.384139,0.015338,0.000384,0.014953,1.027214
2,2006-02-28,443,optimal,0.006213,0.235843,0.009004,0.000236,0.008769,1.036221
3,2006-03-31,441,optimal,0.006824,0.246619,-0.015372,0.000247,-0.015619,1.020037
4,2006-04-30,440,optimal,0.007074,0.172051,-0.016889,0.000172,-0.017061,1.002634


In [6]:
backtest.tail()

,month_end,n_assets,solver_status,objective_value,turnover,gross_return,cost,net_return,cumulative_nav
223,2024-07-31,417,optimal,0.003476,0.063929,0.070253,0.000064,0.070189,5.037283
224,2024-08-31,415,optimal,0.003906,0.061799,0.034928,0.000062,0.034866,5.212915
225,2024-09-30,417,optimal,0.004688,0.284512,-0.039041,0.000285,-0.039326,5.007913
226,2024-10-31,416,optimal,0.004053,0.442008,0.067097,0.000442,0.066655,5.341716
227,2024-11-30,416,optimal,0.006211,0.119563,-0.065253,0.000120,-0.065372,4.992516


## Summary

In [7]:
print(summary_path.read_text())

Static Allocator Baseline v1 Summary

Date range: 2005-12-31 to 2024-11-30
lambda-risk: 10.0
tau-turnover: 0.001
cost-bps: 10.0
Number of attempted months: 228
Number of successful months: 228
Mean n_assets: 419.14
Mean turnover: 0.196101
Mean monthly gross return: 0.007969
Mean monthly net return: 0.007773
Annualized gross return: 0.090868
Annualized net return: 0.088313
Annualized net volatility: 0.128506
Sharpe ratio: 0.725872
Cumulative net return: 3.992516
Max drawdown: -0.480402
Fraction of months successfully solved: 1.000000

